# Session 2.2 — Context Management & Chat History

## What We're Building Today

A working Streamlit chat application that:
- Rewrites follow-up queries for better retrieval
- Assembles retrieved chunks into coherent reading order (grouped by document, sorted by index, gap-marked)
- Manages conversation history via truncation to prevent context overflow
- Lets users toggle between naive and filtered retrieval from the sidebar

## Learning Objectives

1. **Explain** why follow-up questions fail in stateless retrieval and how query rewriting solves the problem
2. **Implement** `contextualize_query()` to rewrite follow-up questions using conversation history and an LLM call
3. **Assemble** retrieved chunks into coherent reading order by grouping by source document, sorting by chunk index, and marking gaps
4. **Manage** chat history within context window limits using a message-count sliding-window truncation strategy
5. **Add** a sidebar toggle that lets users switch between naive and filtered retrieval strategies at runtime

## Session Theme

> "Retrieval is stateless. Your user is not."

In [ ]:
# Path setup — run this cell FIRST
# Notebooks run from their subdirectory but pipeline modules are at the project root
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd().parent.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

# Load environment variables from project root
from dotenv import load_dotenv
load_dotenv(_PROJECT_ROOT / ".env")

print(f"Project root: {_PROJECT_ROOT}")
print(f"Path configured: pipeline imports will work")

In [ ]:
# Diagram rendering helper — run once, use throughout the notebook
from IPython.display import display, HTML

def mermaid(diagram):
    """Render a Mermaid diagram inline in the notebook."""
    display(HTML(f"""
    <script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
    <script>mermaid.initialize({{startOnLoad:true, theme:'neutral'}});</script>
    <div class="mermaid">{diagram}</div>
    """))

In [ ]:
# Import verification
from pipeline.retrieval.naive import naive_retrieve
from pipeline.retrieval.filtered import filtered_retrieve
from pipeline.generation.generate import call_claude
from pipeline.context.assembler import assemble_context, contextualize_query
from pipeline.context.manager import manage_history

print("All pipeline imports successful!")
print("  - pipeline.retrieval.naive")
print("  - pipeline.retrieval.filtered")
print("  - pipeline.generation.generate")
print("  - pipeline.context.assembler")
print("  - pipeline.context.manager")

In [ ]:
# ChromaDB verification
from pipeline.ingestion.store import get_collection

collection = get_collection()
count = collection.count()
print(f"ChromaDB collection: {collection.name}")
print(f"Documents loaded: {count} chunks")
assert count > 0, "No chunks found! Did you run ingestion?"

In [ ]:
# Claude API test
response = call_claude("Say hello in 3 words")
print(f"Claude says: {response}")

In [ ]:
# Verify upgraded rag.py works
from app.rag import get_response, ChatResponse

test_response = get_response("What does Northbrook do?", messages=[])
print(f"Answer: {test_response.answer[:100]}...")
print(f"Sources: {len(test_response.sources)} chunks retrieved")
print(f"Type: {type(test_response).__name__}")

### Setup Checklist

- [ ] All pipeline imports successful
- [ ] ChromaDB collection has chunks loaded
- [ ] Claude API responds
- [ ] Upgraded `app/rag.py` works (returns ChatResponse with answer and sources)

If anything above failed, run `git pull` to get the latest `app/rag.py` upgrade.

In [ ]:
# How the pieces fit together
mermaid("""
graph LR
    U["User"] -->|types question| ST["Streamlit UI<br/>(main.py)"]
    ST -->|question + history| RAG["rag.py<br/>get_response()"]
    RAG -->|rewritten query| CHROMA["ChromaDB"]
    CHROMA -->|ranked chunks| RAG
    RAG -->|assembled prompt| CLAUDE["Claude API"]
    CLAUDE -->|answer| RAG
    RAG -->|ChatResponse| ST
    ST -->|answer + sources| U
    style RAG fill:#e1f0ff,stroke:#4a90d9
    style CHROMA fill:#f0ffe1,stroke:#5a9f4a
    style CLAUDE fill:#ffe1f0,stroke:#d94a7a
""")

In [ ]:
# The 6-step pipeline inside get_response()
mermaid("""
graph TD
    Q["User Question + History"] --> S1
    S1["1. manage_history<br/>Trim to budget"] --> S2
    S2["2. contextualize_query<br/>Rewrite follow-ups"] --> S3
    S3["3. retrieve<br/>Embed → ChromaDB"] --> S4
    S4["4. assemble_context<br/>Group, sort, gaps"] --> S5
    S5["5. Build Prompt<br/>History + Sources + Question"] --> S6
    S6["6. call_claude<br/>Generate answer"]
    S6 --> R["ChatResponse"]
    style S1 fill:#fff3cd,stroke:#856404
    style S2 fill:#d4edda,stroke:#155724
    style S3 fill:#cce5ff,stroke:#004085
    style S4 fill:#d4edda,stroke:#155724
    style S5 fill:#f8d7da,stroke:#721c24
    style S6 fill:#cce5ff,stroke:#004085
""")

---

## The Follow-up Problem

Retrieval is **stateless** — it embeds a single query string. It does not see conversation history.

When a user asks a follow-up question like "How many days do I get?" after discussing vacation policy, the retrieval system embeds exactly those six words. It has no idea the user is talking about vacation days.

```
WHAT THE USER MEANT:
  "How many vacation days does a Northbrook employee receive?"

WHAT RETRIEVAL SAW:
  "How many days do I get?"
```

Let's see this failure in action.

In [ ]:
# Demo: Stateless retrieval failure

# Question 1: Standalone — retrieves well
print("=" * 60)
print("Question 1: 'What is the vacation policy at Northbrook?'")
print("=" * 60)
results_1 = naive_retrieve("What is the vacation policy at Northbrook?", top_k=3)
for r in results_1:
    print(f"  [{r['score']:.3f}] {r['metadata']['source']}: {r['text'][:60]}...")

print()

# Question 2: Follow-up — retrieves poorly
print("=" * 60)
print("Question 2 (follow-up): 'How many days do I get?'")
print("=" * 60)
results_2 = naive_retrieve("How many days do I get?", top_k=3)
for r in results_2:
    print(f"  [{r['score']:.3f}] {r['metadata']['source']}: {r['text'][:60]}...")

print()
print("Notice the score drop on the follow-up question.")

### The Fix: Rewrite the Query Before Retrieval

Claude can read the conversation history and understand what the user means. So before we send the query to retrieval, we ask Claude to rewrite it.

"How many days do I get?" becomes "How many vacation days does a Northbrook employee receive?" — and that retrieves perfectly.

```
RETRIEVAL SCORES:
  Original follow-up:  0.43, 0.41, 0.38  (bad)
  Rewritten query:     0.84, 0.81, 0.78  (good)
```

One extra Claude call before retrieval. The cost: ~0.3 seconds and a few hundred tokens. The benefit: follow-up questions actually work.

In [ ]:
# Data flow: query rewriting
mermaid("""
graph LR
    H["Conversation<br/>History"] --> FMT["Format last<br/>6 messages"]
    UQ["Follow-up<br/>Question"] --> FMT
    FMT --> LLM["Claude<br/>temp=0.0"]
    LLM --> SQ["Standalone<br/>Query"]
    SQ --> EMB["embed_texts()"]
    EMB --> DB["ChromaDB"]
    DB --> CHUNKS["Ranked Chunks"]
    style LLM fill:#ffe1f0,stroke:#d94a7a
    style DB fill:#f0ffe1,stroke:#5a9f4a
    style SQ fill:#e1f0ff,stroke:#4a90d9
""")

---

## Query Rewriting: `contextualize_query()`

Open `pipeline/context/assembler.py` and study `contextualize_query()`. It rewrites follow-up questions using conversation history and an LLM call.

**The logic:**
1. If there is no history (first message), return the query unchanged
2. Format the recent history (last 3-4 exchanges) as a string
3. Call Claude with a rewriting prompt
4. Return the rewritten query

**The rewriting prompt template:**

```
Given the conversation history below, rewrite the user's latest
message as a standalone question suitable for a search engine.
If the message is already standalone, return it unchanged.

Return ONLY the rewritten question — no explanation, no preamble.

History:
{recent_history}

Latest message: {user_message}

Rewritten question:
```

**Key design decisions:**
- Limit history to the last 6-8 messages (3-4 exchanges)
- Use `temperature=0.0` for deterministic output
- Use `call_claude()` from `pipeline.generation.generate`

In [ ]:
# Read the canonical implementation
import inspect
from pipeline.context.assembler import contextualize_query

print(inspect.getsource(contextualize_query))

> **Study this implementation.** Notice three design decisions:
> 1. **Early return** — first question has no history, so skip the LLM call
> 2. **Recency window** — only last 6 messages go to the rewriter
> 3. **Directive prompt** — "Return ONLY the rewritten question" prevents explanations
>
> **Lab 2:** The rewriting prompt in `assembler.py` is marked `CUSTOMIZABLE`.

In [ ]:
# Test your contextualize_query() implementation
from pipeline.context.assembler import contextualize_query

# Simulate a conversation about vacation policy
fake_history = [
    {"role": "user", "content": "What is the vacation policy at Northbrook?"},
    {"role": "assistant", "content": "Northbrook provides employees with paid time off based on tenure..."}
]

# This follow-up should be rewritten to include "vacation"
follow_up = "How many days do I get?"
rewritten = contextualize_query(fake_history, follow_up)

print(f"Original:  {follow_up}")
print(f"Rewritten: {rewritten}")
print()

# Test: first message (no history) should return unchanged
standalone = contextualize_query([], "What is the vacation policy?")
print(f"Standalone (no history): {standalone}")
print()

# Verify the rewritten query retrieves better
print("Retrieval with rewritten query:")
results = naive_retrieve(rewritten, top_k=3)
for r in results:
    print(f"  [{r['score']:.3f}] {r['text'][:60]}...")

### What Did You Notice?

- Does the rewritten query include the topic from the conversation ("vacation")?
- Are the retrieval scores higher with the rewritten query?
- What happens when the message is already standalone?
- How fast was the rewriting call? (Should be <1 second)

In [ ]:
# Data flow: context assembly
mermaid("""
graph TD
    IN["Retrieved Chunks<br/>(similarity order)"] --> GRP["Group by<br/>source document"]
    GRP --> SORT["Sort each group<br/>by chunk_index"]
    SORT --> GAP{"Gap in indices?"}
    GAP -->|yes| MARK["Insert gap marker"]
    GAP -->|no| JOIN["Append chunk"]
    MARK --> JOIN
    JOIN --> HDR["Add source headers"]
    HDR --> OUT["Coherent Context"]
    style IN fill:#cce5ff,stroke:#004085
    style OUT fill:#d4edda,stroke:#155724
    style GAP fill:#fff3cd,stroke:#856404
""")

---

## Context Assembly: `assemble_context()`

Right now, chunks go into the system prompt sorted by cosine similarity — highest score first. This is like ripping pages out of two different books, shuffling them by relevance, and handing someone the stack.

`assemble_context()` reorganizes chunks into **reading order** — same chunks, zero additional API cost, better answers.

```
BEFORE (similarity order — what ChromaDB returns):
  q3-financial.md chunk 5 (0.89) -> board-meeting.md chunk 3 (0.85) ->
  q3-financial.md chunk 2 (0.81) -> board-meeting.md chunk 1 (0.77) ->
  q3-financial.md chunk 4 (0.72)

AFTER (assemble_context — what Claude reads):

  --- Source: q3-financial.md ---
  [chunk 2] "...this report covers Q3 financial performance..."
    [...content omitted...]
  [chunk 4] "...operating expenses remained within budget..."
  [chunk 5] "...Q3 revenue totaled $4.2M, exceeding the..."

  --- Source: board-meeting.md ---
  [chunk 1] "...the board convened to review quarterly..."
    [...content omitted...]
  [chunk 3] "...projections for Q3 were discussed at..."
```

**Three things happen:**
1. Chunks are **grouped** by source document
2. Within each group, chunks are **sorted** by chunk index (original document order)
3. When there is a gap in the index (non-consecutive), insert a **gap marker**

**The algorithm:**
```
assemble_context(chunks):
  1. Group chunks by source document (metadata['source'])
  2. Sort each group by chunk index (metadata['chunk_index'])
  3. For each group:
     a. Write the document header: "--- Source: {source} ---"
     b. For each chunk: if not consecutive with previous, insert gap marker
     c. Write the chunk text
  4. Join groups and return
```

In [ ]:
# Read the canonical implementation
import inspect
from pipeline.context.assembler import assemble_context

print(inspect.getsource(assemble_context))

> **Study this implementation.** The algorithm:
> 1. **Group** — `defaultdict(list)` groups chunks by `metadata['source']`
> 2. **Sort** — each group sorted by `chunk_index`, not globally
> 3. **Gap detection** — if `current_index != prev_index + 1`, insert marker
>
> **Key insight:** Same chunks, zero additional API cost. Only presentation changes.
>
> **Lab 2:** The assembly layout in `assembler.py` is marked `CUSTOMIZABLE`.

In [ ]:
# Test your assemble_context() implementation
from pipeline.context.assembler import assemble_context

# Retrieve some chunks
chunks = naive_retrieve("What is the vacation policy at Northbrook?", top_k=5)

# Show what ChromaDB returns (similarity order)
print("BEFORE — Similarity order from ChromaDB:")
print("-" * 50)
for i, c in enumerate(chunks):
    source = c['metadata'].get('source', 'Unknown')
    idx = c['metadata'].get('chunk_index', '?')
    print(f"  {i+1}. [{c['score']:.3f}] {source} (chunk {idx}): {c['text'][:50]}...")

print()

# Run through assemble_context
assembled = assemble_context(chunks)
print("AFTER — assemble_context() output:")
print("-" * 50)
print(assembled)

### What Did You Notice?

- Are chunks now grouped by source document?
- Within each group, are they in reading order (sorted by chunk index)?
- Do you see gap markers where chunks are not consecutive?
- Same chunks, same cost — but now Claude reads them in coherent order.

In [ ]:
# Data flow: history management
mermaid("""
graph LR
    ALL["All Messages"] --> CHECK{"len > max?"}
    CHECK -->|no| PASS["Return all"]
    CHECK -->|yes| WIN["Sliding Window<br/>messages[-max:]"]
    WIN --> TRIM["Trimmed History"]
    TRIM --> RW["Feeds rewriter"]
    TRIM --> PR["Feeds prompt"]
    style CHECK fill:#fff3cd,stroke:#856404
    style TRIM fill:#d4edda,stroke:#155724
""")

---

## Token Management: `manage_history()`

### Context Window Anatomy

Claude Sonnet 4.5 has a 200,000 token context window. Here is how it fills up:

```
CONTEXT WINDOW ANATOMY (per turn):

System prompt (instructions)     ~500 tokens
Assembled context (5 chunks)     ~640 tokens (128 tokens/chunk x 5)
Chat history (per message)       ~100-500 tokens each
Current question                 ~50-100 tokens
----------------------------------------------------
Reserved for generation          remaining tokens
```

Over a long conversation:

```
After 10 messages:   ~1,500 history + ~1,140 fixed = ~2,640 total
After 30 messages:   ~4,500 history + ~1,140 fixed = ~5,640 total
After 100 messages:  ~15,000 history + ~1,140 fixed = ~16,140 total
```

Two things matter more than the hard limit: **cost** (every token costs money) and **quality degradation** (flooding context with old history dilutes important information).

### Truncation vs Summarization

```
| Strategy      | Cost         | Speed    | Early Context | Complexity    | Best For          |
|---------------|--------------|----------|---------------|---------------|-------------------|
| Truncation    | Free (list)  | Instant  | Lost          | 2 lines       | Short sessions    |
| Summarization | Extra API    | 1-2 sec  | Preserved     | ~30 lines     | Long conversations|
```

Today you implement **truncation**. For the Final Project, summarization is a strong upgrade.

### Why Simple Message Count (Not Token Counting)

Token counting adds an API call per turn. For a classroom app with a 200k context window, that overhead is not worth it. Message-count truncation is simple, predictable, and sufficient. A 10-message window is roughly 1,500-5,000 tokens — well within budget.

In [ ]:
# Read the canonical implementation
import inspect
from pipeline.context.manager import manage_history

print(inspect.getsource(manage_history))

> **Study this implementation.** Two lines prevent context overflow:
> 1. **Even rounding** — `max_messages` rounded down so user/assistant pairs stay together
> 2. **Sliding window** — `messages[-max_messages:]` keeps the most recent
>
> **Lab 2:** The history strategy in `manager.py` is marked `CUSTOMIZABLE`. Alternatives: keep-first-plus-last, summarization, token-based budget.

In [ ]:
# Test your manage_history() implementation
from pipeline.context.manager import manage_history

# Create a fake 15-message conversation
fake_messages = []
for i in range(15):
    role = "user" if i % 2 == 0 else "assistant"
    fake_messages.append({"role": role, "content": f"Message {i+1} from {role}"})

print(f"Total messages: {len(fake_messages)}")
print()

# Truncate to 10
truncated = manage_history(fake_messages, max_messages=10)
print(f"After truncation (max=10): {len(truncated)} messages")
print(f"First kept: {truncated[0]['content']}")
print(f"Last kept:  {truncated[-1]['content']}")
print()

# Under the limit — should return all
short = [{"role": "user", "content": "Hi"}]
result = manage_history(short, max_messages=10)
print(f"Short list (1 message, max=10): {len(result)} messages — unchanged")
print()

# Verify messages 1-5 are dropped, 6-15 remain
assert len(truncated) == 10, f"Expected 10, got {len(truncated)}"
assert truncated[0]["content"] == "Message 6 from assistant", "Wrong first message after truncation"
print("All assertions passed!")

---

## Lab 2 Preview: 7 Customization Points

After Session 3.1, `app/rag.py` becomes yours. It has **7 marked sections** — each is a decision you can measure and defend.

| # | Section | What You Can Change | Example |
|---|---------|-------------------|--------|
| 1 | **Retrieval Strategy** | Which retrieve function to import | `naive_retrieve` → `hyde_retrieve` or `enriched_retrieve` |
| 2 | **System Prompt** | Persona, tone, grounding rules | Tighten grounding, add domain instructions |
| 3 | **History Management** | How history is trimmed | Keep-first-plus-last, summarization |
| 4 | **Query Rewriting** | How follow-ups are rewritten | Change prompt, adjust history window |
| 5 | **Retrieval Parameters** | `top_k`, score thresholds | Retrieve 3 vs 10 chunks, filter by min score |
| 6 | **Context Assembly** | How chunks are presented | Change grouping, add metadata, modify gaps |
| 7 | **Generation Settings** | Model, temperature, max_tokens | Haiku for speed, temperature for variety |

**Lab 2 scope:** Choose at least one retrieval strategy change. Build a golden test set. Produce before/after evidence. Present your findings.

In [ ]:
# See the actual prompt Claude receives
from app.rag import get_response, _SYSTEM_PROMPT
from pipeline.context.assembler import assemble_context, contextualize_query
from pipeline.context.manager import manage_history
from pipeline.retrieval.naive import naive_retrieve

history = [
    {"role": "user", "content": "What is the vacation policy at Northbrook?"},
    {"role": "assistant", "content": "Northbrook provides 20 vacation days per year for full-time employees..."}
]
question = "How many days do I get?"

managed = manage_history(history)
rewritten = contextualize_query(managed, question)
sources = naive_retrieve(rewritten, top_k=5)
assembled = assemble_context(sources)

sections = []
recent = managed[-6:] if managed else []
if recent:
    conv_lines = [f"{'User' if m['role']=='user' else 'Assistant'}: {m['content']}" for m in recent]
    sections.append("## Conversation So Far\n" + "\n\n".join(conv_lines))
sections.append("## Sources\n" + assembled)
sections.append("\n## Current Question\n" + question)
user_message = "\n\n".join(sections)

print("=" * 60)
print("SYSTEM PROMPT:")
print("=" * 60)
print(_SYSTEM_PROMPT)
print("\n" + "=" * 60)
print("USER MESSAGE (what Claude sees):")
print("=" * 60)
print(user_message)
print(f'\nRewritten query for retrieval: "{rewritten}"')

In [ ]:
# The structure of what Claude receives
mermaid("""
graph TD
    SYS["System Prompt<br/><i>Persona + grounding rules</i>"] --> CTX["Claude Context Window"]
    subgraph USER["User Message (built by rag.py)"]
        HIST["## Conversation So Far<br/><i>Last 3 exchanges</i>"]
        SRC["## Sources<br/><i>Grouped, sorted chunks</i>"]
        CUR["## Current Question<br/><i>Original question</i>"]
    end
    HIST --> CTX
    SRC --> CTX
    CUR --> CTX
    CTX --> ANS["Generated Answer"]
    style SYS fill:#ffe1f0,stroke:#d94a7a
    style HIST fill:#fff3cd,stroke:#856404
    style SRC fill:#d4edda,stroke:#155724
    style CUR fill:#cce5ff,stroke:#004085
""")

---

## Integration: End-to-End Verification

Let's verify the full pipeline works together: query rewriting + context assembly + history management.

In [ ]:
# Full pipeline test: ask a question, get a response with sources
from app.rag import get_response

# First question — standalone
response = get_response("What is the vacation policy at Northbrook?", messages=[])
print("Question: What is the vacation policy at Northbrook?")
print(f"Answer: {response.answer[:200]}...")
print(f"Sources: {len(response.sources)} chunks")
print(f"Rewritten query: {response.rewritten_query}")
print()

# Verify context assembly is working
assembled = assemble_context(response.sources)
print("Context assembly output (first 300 chars):")
print(assembled[:300])

In [ ]:
# Follow-up test: verify query rewriting improves retrieval scores
from app.rag import get_response

# Simulate conversation history
history = [
    {"role": "user", "content": "What is the vacation policy at Northbrook?"},
    {"role": "assistant", "content": "Northbrook provides paid time off based on employee tenure..."}
]

# Follow-up question
follow_up_response = get_response("How many days do I get?", messages=history)

print("Follow-up: 'How many days do I get?'")
print(f"Rewritten to: {follow_up_response.rewritten_query}")
print()
print(f"Answer: {follow_up_response.answer[:200]}...")
print()
print("Source scores (should be 0.7+ if rewriting works):")
for s in follow_up_response.sources[:3]:
    print(f"  [{s['score']:.3f}] {s['metadata'].get('source', 'unknown')}")

---

## What You Built Today

- **Query Rewriting** (`contextualize_query()`) — follow-up questions are rewritten as standalone queries before retrieval
- **Context Assembly** (`assemble_context()`) — retrieved chunks are grouped by document, sorted by index, gap-marked
- **History Management** (`manage_history()`) — conversation history is truncated to prevent context overflow
- **Retrieval Toggle** — sidebar radio button switches between naive and filtered strategies

### Verification Checklist

- [ ] Follow-up questions retrieve well (scores 0.7+ instead of 0.4)
- [ ] No crashes after 10+ messages
- [ ] Sources display in expander below each response
- [ ] Sidebar toggle switches between naive and filtered

### Questions to Leave With

> **On Trust:** "You built an app that answers questions from your corpus. What if a user tries to trick it?"

> **On Safety:** "What is the worst thing a user could make your app do?"

---

### Next Session: 3.1 — Prompt Injection & Defensive Design

- What happens when someone tries to break your app?
- "Ignore all previous instructions" — prompt injection attacks
- Defensive strategies and grounding with source material

**Lab 1 Reminder:** Due end of Session 3.2. Your GUI customizations are untouched by today's backend work — different files entirely.